Similarity Search using FAISS on 512 dim cover photo embedding vectors

In [ ]:
from tqdm import tqdm
import faiss
import numpy as np

from src.features import build_features


def load_to_disk():
    xb = np.load(r"D:\Grailed_Npys\embeddings.npy", mmap_mode='r')
    d = 512
    res = faiss.StandardGpuResources()
    cpu_index = faiss.IndexFlatIP(d)
    index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
    index.add(xb)
    k = 100
    mmi = np.lib.format.open_memmap(r'D:\Grailed_Npys\FAISS\similarity_search_neighbours', mode='w+', dtype=np.int64, shape=(len(xb), k))
    mmd = np.lib.format.open_memmap(r'D:\Grailed_Npys\FAISS\similarity_search_distances', mode='w+', dtype=np.float16, shape=(len(xb), k))
    batch_size = 50000
    for i in tqdm(range(0, len(xb), batch_size)):
        end = min(i + batch_size, len(xb))
        D, I = index.search((xb[i:end]), k)
        mmi[i:end] = I
        mmd[i:end] = D
    mmi.flush()
    mmd.flush()

load_to_disk()

In [21]:
import polars as pl
import numpy as np
from src.features import build_features

def similarity_search():
    df = pl.scan_parquet('../data/parquets/sold_listings_20260901.parquet').lazy()
    df = build_features(df)
    ids = np.load(r'D:\Grailed_Npys\FAISS\similarity_search_neighbours', mmap_mode='r')
    query_ids = np.arange(len(ids)) + 1
    print(len(query_ids), len(np.unique(query_ids)))
    neighbor_ids = ids + 1
    distanes = np.load(r'D:\Grailed_Npys\FAISS\similarity_search_distances', mmap_mode='r')
    Ids_dists_df = pl.LazyFrame({
        'query_ids': query_ids,
        'neighbor_ids': neighbor_ids,
        'distances': distanes
    })
    joined_df = df.join(Ids_dists_df, left_on='id', right_on='query_ids', how='left')
    joined_df = joined_df.explode(['neighbor_ids', 'distances'], empty_as_null=True)
    joined_df = joined_df.filter((pl.col('distances') > 0.85))
    neighbors_df = df.select('id', 'category', 'primary_designer', 'log_sold_price')
    print(neighbor_ids.min(), neighbor_ids.max())
    joined_df = joined_df.join(neighbors_df, left_on='neighbor_ids', right_on='id', how='left')
    df = ((joined_df
          .filter((pl.col('neighbor_ids') != pl.col('id')) & (pl.col('category') == pl.col('category_right')) & (pl.col('primary_designer') == pl.col('primary_designer_right'))))
          .group_by('id')
          .agg(pl.col('log_sold_price_right').std().alias('std'), pl.len().alias('similar_count'), pl.col('neighbor_ids')))
    print(df.filter(pl.col('similar_count') > 20).head(10).collect())
    return df
dff = similarity_search()

4345274 4345274
1 4345274
shape: (10, 4)
┌─────────┬──────────┬───────────────┬───────────────────────────────┐
│ id      ┆ std      ┆ similar_count ┆ neighbor_ids                  │
│ ---     ┆ ---      ┆ ---           ┆ ---                           │
│ i64     ┆ f64      ┆ u32           ┆ list[i64]                     │
╞═════════╪══════════╪═══════════════╪═══════════════════════════════╡
│ 1987964 ┆ 0.420819 ┆ 97            ┆ [4064306, 4065165, … 4058942] │
│ 2816400 ┆ 0.686013 ┆ 91            ┆ [686504, 683794, … 690804]    │
│ 2309514 ┆ 0.421204 ┆ 40            ┆ [78365, 2304422, … 2304210]   │
│ 1602328 ┆ 0.565131 ┆ 23            ┆ [3641193, 1599267, … 1609076] │
│ 1088889 ┆ 0.538757 ┆ 72            ┆ [1098031, 1088286, … 1101918] │
│ 1100822 ┆ 0.336408 ┆ 30            ┆ [3156309, 1097532, … 3143514] │
│ 803522  ┆ 0.487794 ┆ 69            ┆ [2911321, 910037, … 773974]   │
│ 1024261 ┆ 0.364489 ┆ 99            ┆ [3118372, 1023421, … 1018349] │
│ 860022  ┆ 0.442391 ┆ 87           

In [22]:
print(dff.filter(pl.col('similar_count') >= 3).select(pl.col('std').mean()).collect())

shape: (1, 1)
┌──────────┐
│ std      │
│ ---      │
│ f64      │
╞══════════╡
│ 0.403103 │
└──────────┘
